# Predicting Card Mana Value in the MTG Star Trek Set
## 1. Project Setup and Data Ingestion
This notebook executes the data pipeline and Random Forest classification model outlined in the PACE Strategy Document. The goal is to predict a card's balanced mana cost based on its base stats and unstructured ability text using Natural Language Processing (NLP).

In [20]:
import pandas as pd
import requests
import time

headers = {'User-Agent': 'MTG_Data_Project/1.0'}

# Search for the main set (trk) AND the supplementary/Commander set (trkc)
# Scryfall groups by unique card name by default, preventing duplicate alternate arts
url = "https://api.scryfall.com/cards/search?q=s:trk,trc&unique=prints"

all_cards = []
has_more = True

print("Fetching data from Scryfall...")

while has_more:
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        all_cards.extend(data['data']) # Add this page's cards to our master list
        
        has_more = data.get('has_more', False) # Check if there is another page
        if has_more:
            url = data['next_page'] # Overwrite the URL with the next page link
            time.sleep(0.1) # Best practice: pause 100ms between requests
    else:
        print(f"Error {response.status_code}: {response.json().get('details', 'Failed')}")
        break

print(f"Successfully downloaded {len(all_cards)} unique cards.")

Fetching data from Scryfall...
Successfully downloaded 1142 unique cards.


## 2. Data Loading & Initial Exploration
With the raw JSON data downloaded, the next step is to load it into a structured Pandas DataFrame. This allows inspection of the columns provided by the Scryfall API and identify which features (like mana cost, power, toughness, and oracle text) needed to isolate for the Random Forest model.

In [21]:
# Convert the list of JSON card dictionaries into a Pandas DataFrame
df = pd.DataFrame(all_cards)

# Display the datatypes
print(f"Dataframe information: {df.info(verbose=True, max_cols=None)}")

# Display the first 5 rows to inspect the data structure
pd.set_option('display.max_columns', None)
df.head()


<class 'pandas.DataFrame'>
RangeIndex: 1142 entries, 0 to 1141
Data columns (total 73 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   object             1142 non-null   str    
 1   id                 1142 non-null   str    
 2   oracle_id          1142 non-null   str    
 3   multiverse_ids     1142 non-null   object 
 4   resource_id        1049 non-null   str    
 5   mtgo_id            1004 non-null   float64
 6   tcgplayer_id       1139 non-null   float64
 7   cardmarket_id      650 non-null    float64
 8   name               1142 non-null   str    
 9   lang               1142 non-null   str    
 10  released_at        1142 non-null   str    
 11  uri                1142 non-null   str    
 12  scryfall_uri       1142 non-null   str    
 13  layout             1142 non-null   str    
 14  highres_image      1142 non-null   bool   
 15  image_status       1142 non-null   str    
 16  image_updated_at   1142 non-null   

,object,id,oracle_id,multiverse_ids,resource_id,mtgo_id,tcgplayer_id,cardmarket_id,name,lang,released_at,uri,scryfall_uri,layout,highres_image,image_status,image_updated_at,image_uris,mana_cost,cmc,type_line,oracle_text,colors,color_identity,keywords,legalities,games,reserved,game_changer,foil,nonfoil,finishes,oversized,promo,reprint,variation,set_id,set,set_name,set_type,set_uri,set_search_uri,scryfall_set_uri,rulings_uri,prints_search_uri,collector_number,digital,rarity,flavor_text,card_back_id,artist,artist_ids,illustration_id,border_color,frame,full_art,textless,booster,story_spotlight,promo_types,edhrec_rank,prices,related_uris,purchase_uris,penny_rank,power,toughness,all_parts,produced_mana,frame_effects,security_stamp,preview,flavor_name
0,card,02080f42-863a-471b-992b-67e35ef1b7b7,9356aee1-3d2b-4d17-8863-663e1ddf56f3,[],5C4FB9C107C8D8B2D4297751D95B760C1E36C6961FA9A1...,158142.0,728620.0,917666.0,"70,000 Light-Years from Home",en,2026-11-13,https://api.scryfall.com/cards/02080f42-863a-4...,https://scryfall.com/card/trk/1/70000-light-ye...,normal,True,highres_scan,2026-08-06T08:43:00Z,{'small': 'https://cards.scryfall.io/small/fro...,{1}{W},2.0,Sorcery,Exile target creature or Spacecraft. Its contr...,[W],[W],[],"{'standard': 'legal', 'future': 'legal', 'hist...","[paper, mtgo, arena]",False,False,True,True,"[nonfoil, foil]",False,False,False,False,b47039dc-da5a-448c-8feb-e77d458108a6,trk,Star Trek,expansion,https://api.scryfall.com/sets/b47039dc-da5a-44...,https://api.scryfall.com/cards/search?order=se...,https://scryfall.com/sets/trk?utm_source=api,https://api.scryfall.com/cards/02080f42-863a-4...,https://api.scryfall.com/cards/search?order=re...,1,False,uncommon,"""If these sensors are working . . . we're on t...",0aeebaf5-8c7d-4636-9e82-8c27447861f7,Ariel Orea,[acaba9f2-ef15-499c-9660-bf91cab2dbd6],3db3beee-6070-4345-a0b4-c27605a84a01,black,2015,False,False,False,False,[universesbeyond],26744.0,"{'usd': '0.31', 'usd_foil': '0.23', 'usd_etche...",{'tcgplayer_infinite_articles': 'https://partn...,{'tcgplayer': 'https://partner.tcgplayer.com/c...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,card,847b4081-25c9-4a9a-a2f3-d5fdadbfe81b,f9db72dc-9a5b-48a4-a86e-7464d9a2166a,[],6E42748D052EB8220B605524E2960C0F173EB9C3DEB012...,157116.0,726372.0,919573.0,Abrade,en,2026-11-13,https://api.scryfall.com/cards/847b4081-25c9-4...,https://scryfall.com/card/trc/449/abrade?utm_s...,normal,True,highres_scan,2026-10-27T14:38:14Z,{'small': 'https://cards.scryfall.io/small/fro...,{1}{R},2.0,Instant,Choose one —\n• Abrade deals 3 damage to targe...,[R],[R],[],"{'standard': 'legal', 'future': 'legal', 'hist...","[paper, mtgo]",False,False,True,True,"[nonfoil, foil]",False,False,True,False,a38c10be-90b1-4bb9-bd74-46784a751ac3,trc,Star Trek Commander,commander,https://api.scryfall.com/sets/a38c10be-90b1-4b...,https://api.scryfall.com/cards/search?order=se...,https://scryfall.com/sets/trc?utm_source=api,https://api.scryfall.com/cards/847b4081-25c9-4...,https://api.scryfall.com/cards/search?order=re...,449,False,uncommon,Quantum torpedoes channel a matter/antimatter ...,0aeebaf5-8c7d-4636-9e82-8c27447861f7,Bartek Fedyczak,[70b1ba71-4ee1-4087-8a9e-ea919f20ebff],0d2f7b6c-bf79-46e7-a547-368cdc28f225,black,2015,False,False,False,False,"[surgefoil, universesbeyond]",144.0,"{'usd': None, 'usd_foil': None, 'usd_etched': ...",{'tcgplayer_infinite_articles': 'https://partn...,{'tcgplayer': 'https://partner.tcgplayer.com/c...,241.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,card,660ec8bc-dc03-4b42-84f3-091c19e2a853,8802b7f0-1ed1-4872-9939-182afb8441ca,[],6E520F7756A51403038DA83894FD65EEEA177FB22E9D37...,158414.0,726700.0,919806.0,Academy Instructor,en,2026-11-13,https://api.scryfall.com/cards/660ec8bc-dc03-4...,https://scryfall.com/card/trk/137/academy-inst...,normal,True,highres_scan,2026-10-28T11:46:45Z,{'small': 'https://cards.scryfall.io/small/fro...,{2}{R},3.0,Creature — Bolian Officer,This creature gets +1/+0 for each Officer toke...,[R],[R],[],"{'standard': 'legal',

## 3. Feature Selection
The Scryfall API returns 72 columns, but our Random Forest model requires a specific subset to evaluate game balance. Isolating the mechanical identity (`oracle_id`), naming attributes (`name`, `flavor_name`), base stats (`power`, `toughness`), casting properties (`mana_cost`, `cmc`, `colors`), ability text (`oracle_text`), and set context (`type_line`, `rarity`, `reprint`, `legalities`).

In [23]:
# Define the target columns for the predictive model and EDA
columns_to_keep = [
    'oracle_id', 'name', 'flavor_name', 'mana_cost', 'colors', 
    'type_line', 'cmc', 'oracle_text', 'power', 'toughness', 
    'rarity', 'reprint', 'legalities'
]

# Create the focused dataframe
df_model = df[columns_to_keep].copy()

# Display the shape and first few rows to verify
print(f"Focused dataframe shape: {df_model.shape}")
df_model.head()

Focused dataframe shape: (1142, 13)


,oracle_id,name,flavor_name,mana_cost,colors,type_line,cmc,oracle_text,power,toughness,rarity,reprint,legalities
0,9356aee1-3d2b-4d17-8863-663e1ddf56f3,"70,000 Light-Years from Home",NaN,{1}{W},[W],Sorcery,2.0,Exile target creature or Spacecraft. Its contr...,NaN,NaN,uncommon,False,"{'standard': 'legal', 'future': 'legal', 'hist..."
1,f9db72dc-9a5b-48a4-a86e-7464d9a2166a,Abrade,NaN,{1}{R},[R],Instant,2.0,Choose one —\n• Abrade deals 3 damage to targe...,NaN,NaN,uncommon,True,"{'standard': 'legal', 'future': 'legal', 'hist..."
2,8802b7f0-1ed1-4872-9939-182afb8441ca,Academy Instructor,NaN,{2}{R},[R],Creature — Bolian Officer,3.0,This creature gets +1/+0 for each Officer toke...,0,3,uncommon,False,"{'standard': 'legal', 'future': 'legal', 'hist..."
3,a3da7d5b-2c2b-45fe-b9c5-413b8c8fc0a2,Academy Ruins,NaN,,[],Legendary Land,0.0,"{T}: Add {C}.\n{1}{U}, {T}: Put target artifac...",NaN,NaN,rare,True,"{'standard': 'not_legal', 'future': 'not_legal..."
4,a3da7d5b-2c2b-45fe-b9c5-413b8c8fc0a2,Academy Ruins,NaN,,[],Legendary Land,0.0,"{T}: Add {C}.\n{1}{U}, {T}: Put target artifac...",NaN,NaN,rare,True,"{'standard': 'not_legal', 'future': 'not_legal..."


## 4. Deduplication and Legalities Filter
Because the API query pulled all physical prints to capture the Star Trek flavor treatments and Commander cards, the dataset contains mechanically identical alternate arts. To prevent the Random Forest from over-indexing on cards with multiple printings (like Bristly Bill / Leonard McCoy), dropping duplicate rows based on their underlying `oracle_id`.

In [24]:
# Drop mechanically identical cards to prevent model skew, keeping the first occurrence
df_unique = df_model.drop_duplicates(subset=['oracle_id']).copy()

# Display the shape after removing alternate arts
print(f"Dataframe shape after deduplication: {df_unique.shape}")

# Verify how many unique flavor-named cards remain in the training set
flavor_count = df_unique['flavor_name'].notna().sum()
print(f"Unique cards with Star Trek flavor names remaining: {flavor_count}")

df_unique.head()

Dataframe shape after deduplication: (647, 13)
Unique cards with Star Trek flavor names remaining: 4


,oracle_id,name,flavor_name,mana_cost,colors,type_line,cmc,oracle_text,power,toughness,rarity,reprint,legalities
0,9356aee1-3d2b-4d17-8863-663e1ddf56f3,"70,000 Light-Years from Home",NaN,{1}{W},[W],Sorcery,2.0,Exile target creature or Spacecraft. Its contr...,NaN,NaN,uncommon,False,"{'standard': 'legal', 'future': 'legal', 'hist..."
1,f9db72dc-9a5b-48a4-a86e-7464d9a2166a,Abrade,NaN,{1}{R},[R],Instant,2.0,Choose one —\n• Abrade deals 3 damage to targe...,NaN,NaN,uncommon,True,"{'standard': 'legal', 'future': 'legal', 'hist..."
2,8802b7f0-1ed1-4872-9939-182afb8441ca,Academy Instructor,NaN,{2}{R},[R],Creature — Bolian Officer,3.0,This creature gets +1/+0 for each Officer toke...,0,3,uncommon,False,"{'standard': 'legal', 'future': 'legal', 'hist..."
3,a3da7d5b-2c2b-45fe-b9c5-413b8c8fc0a2,Academy Ruins,NaN,,[],Legendary Land,0.0,"{T}: Add {C}.\n{1}{U}, {T}: Put target artifac...",NaN,NaN,rare,True,"{'standard': 'not_legal', 'future': 'not_legal..."
5,37f3eda9-3242-4d62-846f-935543d8f4fd,Adversary by Algorithm,NaN,{2}{G},[G],Sorcery,3.0,Create a 3/3 colorless Illusion artifact creat...,NaN,NaN,uncommon,False,"{'standard': 'legal', 'future': 'legal', 'hist..."
